**Investments: Theory and Data Analysis**, Bates, Boyer, and Fletcher


# Chapter 6: Alpha Vantage Stock Data with `farms`

This notebook introduces the `farms` Alpha Vantage loaders for adjusted monthly and weekly stock prices. The examples use Microsoft (`MSFT`), but the same functions can be used with other supported stock, ETF, or mutual-fund symbols. The examples use endpoints available to free Alpha Vantage accounts.

## Learning objectives

By the end of this notebook, you should be able to:

- Set an Alpha Vantage API key as a Windows user environment variable without placing it in a notebook.
- Use `farms.load_alpha_vantage()` with one ticker or multiple tickers.
- Distinguish raw closing prices from adjusted closing prices.
- Calculate simple returns from adjusted closing prices.
- Recognize API-rate-limit and credential-management issues.

## Data access

The examples require an Alpha Vantage API key and an internet connection. To get a key, visit the [Alpha Vantage API key page](https://www.alphavantage.co/support/#api-key), enter your email address, and follow the instructions to receive your free key. Copy the key exactly as provided.

For permanent Windows setup, you will need to run a command in `PowerShell`, Windows' command-line and scripting program. To open a `PowerShell` session, select the Windows Start menu, type `PowerShell`, and choose `Windows PowerShell` or `PowerShell` from the results. You can also open Windows Terminal and select a PowerShell tab. Then run the following command by copying and pasting at the `PowerShell` prompt. Replace only the placeholder value `your-alpha-vantage-key` with your Alpha Vantage key:

```powershell
[Environment]::SetEnvironmentVariable(
    "ALPHAVANTAGE_API_KEY",
    "your-alpha-vantage-key",
    "User"
)
```

The `User` scope saves the variable for future Windows sessions. Close and reopen this notebook, or restart the kernel so the notebook process can see the new variable. Do not place the key directly in a code cell.

### Google Colab setup

Colab runs in a temporary cloud runtime, so the Windows PowerShell environment variable is not available there. Colab **Secrets** are the recommended permanent setup for Colab: the secret is stored separately from the notebook and remains available after the runtime resets.

To set it up, open the **Secrets** panel using the key icon in the left sidebar, add a secret named `ALPHAVANTAGE_API_KEY`, paste your Alpha Vantage key as its value, and enable notebook access for that secret.

A Colab Secret does not become a Windows environment variable. The notebook retrieves it directly with `userdata.get('ALPHAVANTAGE_API_KEY')`. Secrets are tied to your Colab account and are not shared automatically with people who open a shared notebook; each collaborator must add their own secret.

The Colab and Windows setup methods keep the API key outside the notebook. The installation cell requests `farms` version 0.1.35 or newer because multi-ticker support was added in that release. If Colab says that the kernel must be restarted after installation, choose **Runtime > Restart session** and then run the notebook cells again; Python does not replace a package that is already loaded in memory.

In [3]:
%pip install -qU "farms>=0.1.35"

import os

import farms as fm
import matplotlib.pyplot as plt
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

Note: you may need to restart the kernel to use updated packages.


### Read the API key securely

The `farms.get_alpha_vantage_api_key()` function reads your API key. It supports both Windows and Google Colab environments. The function never prints the key.

In [4]:
api_key = fm.get_alpha_vantage_api_key()

Alpha Vantage API key loaded.


## Use the generalized loader

Use `fm.load_alpha_vantage()` when you want one selected series for one ticker or multiple tickers. A single ticker returns one selected-field column; multiple tickers are fetched and aligned by date, producing one column per ticker. The function supports monthly and weekly data; daily Alpha Vantage data is intentionally not exposed because the adjusted daily endpoint requires premium access.

### Required inputs

- `symbol`: one ticker string, such as `"MSFT"`, or an iterable of ticker strings, such as `["MSFT", "AAPL"]`.
- `api_key`: your Alpha Vantage API key, loaded securely in the preceding cell.
- `frequency`: `"monthly"` or `"weekly"`.
- `field`: `"open"`, `"high"`, `"low"`, `"close"`, or `"returns"`. `"returns"` is the decimal percentage change in adjusted close.

### Optional inputs

- `start_date`: lower inclusive date bound; default is `None`. Use `YYYY-MM` for monthly data and `YYYY-MM-DD` for weekly data.
- `end_date`: upper inclusive date bound; default is `None`. Use the same format as `start_date`.
- `timeout`: request timeout in seconds; default is `30`.
- `max_retries`: number of retries for transient failures and rate-limit responses; default is `3`.
- `backoff_factor`: starting delay between retries; default is `1.0`.
- `session`: an optional `requests.Session`; default is `None`, which uses the standard requests call.

For example, this requests monthly adjusted closing prices for MSFT and AAPL. The returned DataFrame has a `date` index and one column per ticker.

In [5]:
close = fm.load_alpha_vantage(
    symbol=['MSFT', 'AAPL'],
    api_key=api_key,
    frequency='monthly',
    field='close',
    start_date='2020-01',
    end_date='2020-12',
)

close.head()

ValueError: symbol must be a nonempty string.

## Request returns for the same tickers

Change only the `field` argument to request decimal returns calculated from adjusted close. The column names remain the ticker symbols.

In [ ]:
returns = fm.load_alpha_vantage(
    symbol=['MSFT', 'AAPL'],
    api_key=api_key,
    frequency='monthly',
    field='returns',
    start_date='2020-01',
    end_date='2020-12',
)

returns.head()

## Interpret the side-by-side result

The generalized loader makes one Alpha Vantage request per ticker. It then aligns the selected series by date, so the result is convenient for comparing securities. The accepted `field` values are `open`, `high`, `low`, `close`, and `returns`. The `returns` field is calculated from adjusted close and is expressed as a decimal return, so `0.01` means 1%.

Use the frequency-specific date format described below: `YYYY-MM` for monthly data and `YYYY-MM-DD` for weekly data.

In [ ]:
returns.head()

## Output and date conventions

| Loader | Index type | Date bounds |
|---|---|---|
| Monthly | `PeriodIndex` | `YYYY-MM` |
| Weekly | `PeriodIndex` with `W-FRI` frequency | `YYYY-MM-DD` |

All Alpha Vantage results append `Return` as the final column when the full table is requested. It is the decimal percentage change in `Adjusted Close`; the first observation has no prior value and is `NaN`.

Monthly data represents whole calendar months. Weekly data represents weeks ending on Friday.

## Calculate returns from adjusted prices

The adjusted close is useful for a simple historical return calculation because it reflects splits and dividends. The first observation has no previous price, so `pct_change()` produces a missing value there.

In [ ]:
msft_returns = returns['MSFT'].dropna()
msft_returns.describe()

In [ ]:
ax = close['MSFT'].plot(figsize=(10, 4), title='MSFT Monthly Close')
ax.set_xlabel('Date')
ax.set_ylabel('Close')
plt.show()

## Common mistakes and security reminders

- Do not hard-code `ALPHAVANTAGE_API_KEY` in a notebook or commit it to Git.
- If the environment variable is set after Jupyter starts, restart the Jupyter server or kernel.
- Use `YYYY-MM` for monthly bounds and `YYYY-MM-DD` for weekly bounds.
- `Close` is the unadjusted closing price; `Adjusted Close` reflects historical split and dividend adjustments.
- Alpha Vantage can return rate-limit or informational messages instead of time-series data. The `farms` loaders raise clear exceptions for those responses.
- Daily Alpha Vantage data is not supported by these examples because the adjusted daily endpoint requires premium access.

## Try it

1. Replace `MSFT` with another supported symbol and compare its monthly and weekly adjusted prices.
2. Change the monthly date range and inspect the `PeriodIndex`.
3. Add a third ticker to the `symbol` list and compare the resulting columns.
4. Change `field` from `close` to `returns` and compare the resulting series.
5. Compare monthly `Close` and `Adjusted Close` in a full response. When do they differ?
6. Explain why storing an API key in a user environment variable is safer than placing it in a notebook cell.